# Auditory Oddball ERP Pipeline (P300, N200, MMN)

**Author:** Sanjana Singh  
**Topic:** EEG/ERP analysis in MNE-Python

Single-subject 64-channel BioSemi auditory oddball (OpenNeuro ds003061). Full pipeline: bad-channel interpolation, bandpass/notch filtering, ICA artifact removal, average referencing, epoching, P300/N200 statistics, MMN difference wave, and reference-electrode comparison.


## setup

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne
from mne.preprocessing import ICA
from scipy import stats
from scipy.stats import shapiro, mannwhitneyu, ttest_ind

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# plot defaults
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# colors used throughout
C_STD   = '#2166ac'   # standard - blue
C_ODD   = '#d6604d'   # oddball - red
C_NOISE = '#4dac26'   # noise - green
C_DIFF  = '#762a83'   # difference wave - purple

# paths
DATA_DIR        = os.path.expanduser('~/Desktop/eeg')
EEG_FILE        = os.path.join(DATA_DIR, 'sub-001_task-P300_run-1_eeg.set')
EVENTS_FILE     = os.path.join(DATA_DIR, 'sub-001_task-P300_run-1_events.tsv')
CHANNELS_FILE   = os.path.join(DATA_DIR, 'sub-001_task-P300_run-1_channels.tsv')
ELECTRODES_FILE = os.path.join(DATA_DIR, 'sub-001_task-P300_run-1_electrodes.tsv')

FIG_DIR = os.path.expanduser('~/Desktop/figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('mne version:', mne.__version__)
print('data dir:', DATA_DIR)
print('fig dir:', FIG_DIR)


## load .set file

In [2]:
raw = mne.io.read_raw_eeglab(EEG_FILE, preload=True, verbose=False)

# fix channel types - mne reads everything as eeg, but channels.tsv has the real types
ch_type_df = pd.read_csv(CHANNELS_FILE, sep='\t')
bids_to_mne = {
    'EEG': 'eeg', 'MISC': 'misc', 'GSR': 'misc',
    'RESP': 'bio', 'TEMP': 'misc', 'ECG': 'ecg', 'EMG': 'emg',
}
type_corrections = {}
for _, row in ch_type_df.iterrows():
    ch = row['name']
    if ch in raw.ch_names:
        mne_type = bids_to_mne.get(str(row['type']).upper(), 'misc')
        current = raw.get_channel_types([ch])[0]
        if current != mne_type:
            type_corrections[ch] = mne_type
if type_corrections:
    raw.set_channel_types(type_corrections)
    print('fixed channel types:', type_corrections)

print('channels:', raw.info['nchan'])
print('sfreq:', raw.info['sfreq'], 'Hz')
print('duration:', round(raw.times[-1], 1), 's')
print('eeg channels:', len(mne.pick_types(raw.info, eeg=True)))
print('misc channels:', len(mne.pick_types(raw.info, misc=True)))


## load events

events.tsv has string labels - need to map to integer ids for mne.
combining oddball + oddball_with_reponse into one condition,
same for noise. dropping button presses and ignore markers.


In [3]:
events_df = pd.read_csv(EVENTS_FILE, sep='\t')
print('event counts in tsv:')
print(events_df['value'].value_counts().to_string())
print()

label_to_id = {
    'standard': 1,
    'oddball_with_reponse': 2,
    'oddball': 2,
    'noise': 3,
    'noise_with_reponse': 3,
}

events_df = events_df[events_df['value'].isin(label_to_id.keys())].copy()
events_df['event_id'] = events_df['value'].map(label_to_id)

sfreq = raw.info['sfreq']
events_df['sample_idx'] = (events_df['onset'] * sfreq).round().astype(int)
events = events_df[['sample_idx']].copy()
events['prev'] = 0
events['id'] = events_df['event_id'].values
events_array = events[['sample_idx', 'prev', 'id']].to_numpy(dtype=int)

event_id = {'standard': 1, 'oddball': 2, 'noise': 3}

print('mapped event counts:')
for label, eid in event_id.items():
    n = (events_array[:, 2] == eid).sum()
    print(' ', label, ':', n)
print('total:', len(events_array))


## montage from electrodes.tsv

In [4]:
elec_df = pd.read_csv(ELECTRODES_FILE, sep='\t')
print('electrodes.tsv rows:', len(elec_df))
print(elec_df.head(6).to_string())

ch_info = pd.read_csv(CHANNELS_FILE, sep='\t')
eeg_names = ch_info[ch_info['type'] == 'EEG']['name'].tolist()
elec_eeg = elec_df[elec_df['name'].isin(eeg_names)].copy()

try:
    # mm to m
    ch_pos = {row['name']: np.array([row['x'], row['y'], row['z']]) / 1000.0
              for _, row in elec_eeg.iterrows()
              if not any(np.isnan([row['x'], row['y'], row['z']]))}

    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
    raw.set_montage(montage, on_missing='warn')
    print('custom montage set,', len(ch_pos), 'channels')
except Exception as e:
    print('custom montage failed:', e)
    print('using standard_1020 instead')
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')


## raw psd + time series (before any cleaning)

In [5]:
# raw psd
raw_eeg = raw.copy().pick('eeg')

fig, ax = plt.subplots(figsize=(10, 5))
raw_eeg.compute_psd(fmax=60, n_fft=2048).plot(
    axes=ax, show=False, color=C_STD, alpha=0.6, average=True)
ax.set_title('Raw PSD - all EEG channels (averaged)', fontsize=14)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power spectral density (dB/Hz)')
ax.axvline(50, color='gray', linestyle='--', alpha=0.7, label='50 Hz line noise')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '01_raw_psd.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 01_raw_psd.png')

# raw time series, 30-40s, first 10 channels, mean centered for plotting
t_start = 30.0
t_end = t_start + 10.0
n_plot = 10
ch_names_plot = raw_eeg.ch_names[:n_plot]
data_plot = raw_eeg.get_data(picks=ch_names_plot, tmin=t_start, tmax=t_end)
data_time = raw_eeg.times[(raw_eeg.times >= t_start) & (raw_eeg.times < t_end)]

# subtract per-channel mean so the dc offset doesn't blow up the y axis
data_plot = data_plot - data_plot.mean(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 8))
for i, (ch, trace) in enumerate(zip(ch_names_plot, data_plot)):
    ax.plot(data_time, trace * 1e6 + i * 150, color=C_STD, lw=0.6, alpha=0.85)
    ax.text(data_time[-1] + 0.05, i * 150, ch, va='center', fontsize=9)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (uV, mean-centered + offset per channel)')
ax.set_title('Raw EEG - 10-second segment, mean-centered (first 10 channels)', fontsize=14)
ax.set_xlim(data_time[0], data_time[-1] + 0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '01_raw_timeseries.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 01_raw_timeseries.png')


## bad channel detection

using log-variance z-score > 3 to flag broken/flat channels.
log-transform first because raw variance is right-skewed.
cap at 10% so we don't accidentally kill the recording.
then spherical interpolation.


In [6]:
raw_eeg = raw.copy().pick('eeg')
data_arr = raw_eeg.get_data()
ch_names = raw_eeg.ch_names
n_ch = len(ch_names)

log_var = np.log(np.var(data_arr, axis=1) + 1e-30)
lv_med = np.median(log_var)
lv_std = np.std(log_var)
z_scores = np.abs(log_var - lv_med) / (lv_std + 1e-10)

print('top 10 channels by z-score:')
top_idx = np.argsort(z_scores)[::-1][:10]
for i in top_idx:
    print(' ', ch_names[i], ':', round(z_scores[i], 2))

bad_channels = [ch_names[i] for i in range(n_ch) if z_scores[i] > 3.0]

# safety cap at 10%
max_bad = max(1, int(0.10 * n_ch))
if len(bad_channels) > max_bad:
    print('flagged', len(bad_channels), 'channels - capping at', max_bad)
    sort_order = np.argsort(z_scores)[::-1]
    bad_channels = [ch_names[i] for i in sort_order[:max_bad]]

print('bad channels (', len(bad_channels), '):', bad_channels)
raw.info['bads'] = bad_channels

if bad_channels:
    raw.interpolate_bads(reset_bads=True, verbose=False)
    print('interpolated', len(bad_channels), 'channels')
else:
    print('no bad channels found')


## bandpass + notch filter

- bandpass 0.1-30 Hz (FIR, hann window). high-pass kills slow drifts, low-pass kills muscle.
- notch at 50 Hz for india mains. nyquist is 128 Hz so no point doing harmonics.


In [7]:
# keep a copy before filtering for the comparison plot later
raw_prefilt = raw.copy()

raw.filter(l_freq=0.1, h_freq=30.0, method='fir', fir_window='hann',
           picks='eeg', verbose=False)
print('bandpass 0.1-30 Hz done')

raw.notch_filter(freqs=[50.0], picks='eeg', verbose=False)
print('notch 50 Hz done')


## ICA artifact removal

no dedicated EOG was recorded - using Fp1 and Fp2 as eye proxies.
fastica, 15 components, fixed seed.
flag any IC with |pearson r| > 0.3 with either Fp electrode and drop it.


In [8]:
# safety check - if NaN/Inf made it through, ICA will crash
_data_check = raw.copy().pick('eeg').get_data()
if not np.isfinite(_data_check).all():
    print('NaN/Inf found - replacing with zeros')
    _eeg_idx = mne.pick_types(raw.info, eeg=True)
    raw._data[_eeg_idx] = np.nan_to_num(raw._data[_eeg_idx], nan=0.0,
                                         posinf=0.0, neginf=0.0)
else:
    print('data check ok, no NaN/Inf')

# fit ICA
ica = ICA(n_components=15, method='fastica', random_state=42, max_iter=1000)
ica.fit(raw, picks='eeg', verbose=False)
print('ICA fit -', ica.n_components_, 'components')

# correlate each IC with Fp1 and Fp2 to find eye stuff
eog_channels = ['Fp1', 'Fp2']
ic_sources = ica.get_sources(raw).get_data()
ic_sources_z = ic_sources - ic_sources.mean(1, keepdims=True)

exclude_ics = set()
for eog_ch in eog_channels:
    eog_signal = raw.get_data(picks=[eog_ch])[0]
    eog_signal = eog_signal - eog_signal.mean()

    ic_norm = np.linalg.norm(ic_sources_z, axis=1)
    eog_norm = np.linalg.norm(eog_signal)
    corrs = (ic_sources_z @ eog_signal) / (ic_norm * eog_norm + 1e-12)
    high_corr = np.where(np.abs(corrs) > 0.3)[0]
    for ic in high_corr:
        exclude_ics.add(int(ic))
        print(' IC', ic, 'flagged - |r|=', round(abs(corrs[ic]), 3), 'with', eog_ch)

exclude_list = sorted(exclude_ics)
print('ICs to exclude:', exclude_list)

# save IC topo plot
fig_topo = ica.plot_components(picks=None, show=False)
if not isinstance(fig_topo, list):
    fig_topo = [fig_topo]
for fi, fg in enumerate(fig_topo):
    fg.suptitle('ICA Component Topographies', fontsize=13, y=1.01)
    fg.savefig(os.path.join(FIG_DIR, 'ICA_topos_' + str(fi+1) + '.png'),
               dpi=200, bbox_inches='tight')
    plt.close(fg)
print('saved', len(fig_topo), 'IC topo figures')

ica.exclude = exclude_list
ica.apply(raw, verbose=False)
print('ICA applied, removed:', exclude_list)


## re-reference to average

average reference is fine here because the 64 biosemi cap has roughly uniform coverage.
keeping a copy of raw before re-ref for the extra credit section.


In [9]:
raw_before_reref = raw.copy()

raw.set_eeg_reference('average', projection=False, verbose=False)
print('average reference applied')


## post-cleaning psd + time series

In [10]:
raw_clean = raw.copy().pick('eeg')

# clean psd
fig, ax = plt.subplots(figsize=(10, 5))
raw_clean.compute_psd(fmax=60, n_fft=2048).plot(
    axes=ax, show=False, color=C_ODD, alpha=0.6, average=True)
ax.set_title('Post-Preprocessing PSD - all EEG channels (averaged)', fontsize=14)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power spectral density (dB/Hz)')
ax.axvline(50, color='gray', linestyle='--', alpha=0.7, label='50 Hz notch')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '02_clean_psd.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 02_clean_psd.png')

# clean time series
t_start = 30.0
n_plot = 10
ch_names_plot = raw_clean.ch_names[:n_plot]
data_plot = raw_clean.get_data(picks=ch_names_plot, tmin=t_start, tmax=t_start+10)
times_plot = raw_clean.times[(raw_clean.times >= t_start) &
                              (raw_clean.times < t_start+10)]

fig, ax = plt.subplots(figsize=(14, 8))
for i, (ch, trace) in enumerate(zip(ch_names_plot, data_plot)):
    ax.plot(times_plot, trace * 1e6 + i * 150, color=C_ODD, lw=0.6, alpha=0.85)
    ax.text(times_plot[-1] + 0.05, i * 150, ch, va='center', fontsize=9)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (uV, offset per channel)')
ax.set_title('Clean EEG - 10-second segment (first 10 channels)', fontsize=14)
ax.set_xlim(times_plot[0], times_plot[-1] + 0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '02_clean_timeseries.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 02_clean_timeseries.png')


## raw vs clean side by side

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Raw vs Preprocessed EEG Comparison', fontsize=16, fontweight='bold')

raw_raw = raw_prefilt.copy().pick('eeg')

# raw psd
raw_raw.compute_psd(fmax=60, n_fft=2048).plot(
    axes=axes[0, 0], show=False, color=C_STD, alpha=0.7, average=True)
axes[0, 0].set_title('Raw PSD', fontsize=13)
axes[0, 0].axvline(50, color='red', linestyle='--', alpha=0.6, label='50 Hz')
axes[0, 0].legend()

# clean psd
raw_clean.compute_psd(fmax=60, n_fft=2048).plot(
    axes=axes[0, 1], show=False, color=C_ODD, alpha=0.7, average=True)
axes[0, 1].set_title('Clean PSD (filtered + ICA)', fontsize=13)

# raw time series (mean centered)
t_s, t_e = 30.0, 40.0
n_ts = 8
ch_ts = raw_raw.ch_names[:n_ts]
d_raw = raw_raw.get_data(picks=ch_ts, tmin=t_s, tmax=t_e)
d_raw = d_raw - d_raw.mean(axis=1, keepdims=True)
t_ax = raw_raw.times[(raw_raw.times >= t_s) & (raw_raw.times < t_e)]
for i, (ch, tr) in enumerate(zip(ch_ts, d_raw)):
    axes[1, 0].plot(t_ax, tr * 1e6 + i * 200, color=C_STD, lw=0.6)
    axes[1, 0].text(t_ax[-1] + 0.05, i * 200, ch, va='center', fontsize=8)
axes[1, 0].set_title('Raw Time-Series (30-40 s)', fontsize=13)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Amplitude (uV)')

# clean time series
ch_avail = [c for c in ch_ts if c in raw_clean.ch_names]
d_cln = raw_clean.get_data(picks=ch_avail, tmin=t_s, tmax=t_e)
t_axc = raw_clean.times[(raw_clean.times >= t_s) & (raw_clean.times < t_e)]
for i, (ch, tr) in enumerate(zip(ch_avail, d_cln)):
    axes[1, 1].plot(t_axc, tr * 1e6 + i * 200, color=C_ODD, lw=0.6)
    axes[1, 1].text(t_axc[-1] + 0.05, i * 200, ch, va='center', fontsize=8)
axes[1, 1].set_title('Clean Time-Series (30-40 s)', fontsize=13)
axes[1, 1].set_xlabel('Time (s)')

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '03_raw_vs_clean_comparison.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 03_raw_vs_clean_comparison.png')


### what changed

psd: 50 Hz spike gone, sharp rolloff above 30 Hz from the bandpass, less low-freq drift power.
time series: less drift, eye blinks reduced after ICA, alpha rhythm easier to see in the cleaned traces.


## epoching

-200 to 800 ms around each stimulus, baseline = (-200, 0), reject epochs with peak-to-peak > 100 uV on any channel.


In [12]:
tmin, tmax = -0.2, 0.8
baseline = (-0.2, 0.0)
reject_crit = dict(eeg=100e-6)  # 100 uV

epochs = mne.Epochs(
    raw, events_array, event_id=event_id,
    tmin=tmin, tmax=tmax,
    baseline=baseline,
    reject=reject_crit,
    preload=True,
    picks='eeg',
    verbose=False
)
epochs.drop_bad(verbose=False)

print('epoch counts after rejection:')
for cond, eid in event_id.items():
    n = len(epochs[cond])
    print(' ', cond, ':', n)
print('total:', len(epochs))

epochs_std = epochs['standard']
epochs_odd = epochs['oddball']
epochs_nois = epochs['noise']


## evoked + butterfly + GFP per condition

GFP is just std across channels at each time point - shows when scalp-wide activity is biggest.


In [13]:
evoked_std = epochs_std.average()
evoked_odd = epochs_odd.average()
evoked_nois = epochs_nois.average()

evoked_std.comment = 'Standard'
evoked_odd.comment = 'Oddball'
evoked_nois.comment = 'Noise'

def compute_gfp(evoked):
    return evoked.data.std(axis=0)

times_ep = evoked_std.times * 1000  # to ms

cond_info = [
    (evoked_std, C_STD, 'Standard', 'standard'),
    (evoked_odd, C_ODD, 'Oddball', 'oddball'),
    (evoked_nois, C_NOISE, 'Noise', 'noise'),
]

for evk, color, label, fname in cond_info:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                    gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle('ERP: ' + label + ' condition', fontsize=15, fontweight='bold')

    for ch_data in evk.data:
        ax1.plot(times_ep, ch_data * 1e6, color=color, lw=0.4, alpha=0.3)
    ax1.axvline(0, color='k', lw=1, linestyle='--')
    ax1.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax1.set_ylabel('Amplitude (uV)')
    ax1.set_title('Butterfly plot (all channels)')

    gfp = compute_gfp(evk)
    ax2.fill_between(times_ep, 0, gfp * 1e6, color=color, alpha=0.5)
    ax2.plot(times_ep, gfp * 1e6, color=color, lw=1.5)
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('GFP (uV)')
    ax2.set_title('Global Field Power')
    ax2.axvline(0, color='k', lw=1, linestyle='--')

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, 'butterfly_gfp_' + fname + '.png'),
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('saved butterfly_gfp_' + fname + '.png')


## P300

RQ: does the oddball elicit a larger P300 than the standard at Pz?

H0: no difference in mean P300 amplitude (300-500 ms, Pz) between standard and oddball.
H1: oddball P300 at Pz is more positive than standard (one-tailed).


### waveforms at Pz, Cz, CPz

In [14]:
p300_channels = [ch for ch in ['Pz', 'Cz', 'CPz'] if ch in evoked_std.ch_names]
print('plotting P300 at:', p300_channels)

fig, axes = plt.subplots(1, len(p300_channels), figsize=(6 * len(p300_channels), 5),
                          sharey=True)
if len(p300_channels) == 1:
    axes = [axes]

for ax, ch in zip(axes, p300_channels):
    idx_s = evoked_std.ch_names.index(ch)
    idx_o = evoked_odd.ch_names.index(ch)

    ax.plot(times_ep, evoked_std.data[idx_s] * 1e6,
            color=C_STD, lw=2, label='Standard')
    ax.plot(times_ep, evoked_odd.data[idx_o] * 1e6,
            color=C_ODD, lw=2, label='Oddball')

    ax.axvspan(300, 500, alpha=0.12, color='gold', label='P300 window')
    ax.axvline(0, color='k', lw=1, linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_xlabel('Time (ms)', fontsize=12)
    ax.set_ylabel('Amplitude (uV)', fontsize=12)
    ax.set_title('Channel ' + ch, fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(-200, 800)

fig.suptitle('P300 ERP Waveforms: Standard vs Oddball', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '04_P300_erp_waveforms.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 04_P300_erp_waveforms.png')


### topomaps at peak latency

In [15]:
# find P300 peak in oddball at Pz, in the 300-500 ms window
pz_idx = evoked_odd.ch_names.index('Pz')
t_mask_p300 = (evoked_odd.times >= 0.300) & (evoked_odd.times <= 0.500)
p300_data_pz = evoked_odd.data[pz_idx, t_mask_p300]
p300_peak_idx = np.argmax(p300_data_pz)
p300_peak_t = evoked_odd.times[t_mask_p300][p300_peak_idx]
print('P300 peak latency at Pz:', round(p300_peak_t * 1000, 1), 'ms')
print('P300 peak amplitude (oddball):',
      round(evoked_odd.data[pz_idx, t_mask_p300][p300_peak_idx]*1e6, 2), 'uV')

# topomaps
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vlim = max(abs(evoked_std.data[:, t_mask_p300].mean(1).max()),
           abs(evoked_odd.data[:, t_mask_p300].mean(1).max())) * 1e6

for ax, evk, label in zip(axes, [evoked_std, evoked_odd], ['Standard', 'Oddball']):
    mne.viz.plot_topomap(
        evk.data[:, np.argmin(np.abs(evk.times - p300_peak_t))] * 1e6,
        evk.info, axes=ax, show=False, vlim=(-vlim, vlim),
        cmap='RdBu_r', contours=6
    )
    ax.set_title(label + '\n(t = ' + str(int(round(p300_peak_t*1000))) + ' ms)',
                 fontsize=12)

fig.suptitle('P300 Topography at Peak Latency', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '05_P300_topography.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 05_P300_topography.png')


### single-trial amplitude + stats

In [16]:
# single-trial mean amplitude in 300-500 ms at Pz
pz_std_data = epochs_std.copy().pick('Pz').crop(0.300, 0.500).get_data()
pz_odd_data = epochs_odd.copy().pick('Pz').crop(0.300, 0.500).get_data()

p300_std_amp = pz_std_data[:, 0, :].mean(axis=1) * 1e6
p300_odd_amp = pz_odd_data[:, 0, :].mean(axis=1) * 1e6

print('standard: M=', round(p300_std_amp.mean(), 3),
      'SD=', round(p300_std_amp.std(), 3),
      'N=', len(p300_std_amp))
print('oddball:  M=', round(p300_odd_amp.mean(), 3),
      'SD=', round(p300_odd_amp.std(), 3),
      'N=', len(p300_odd_amp))

# normality
_, p_norm_std = shapiro(p300_std_amp[:min(len(p300_std_amp), 5000)])
_, p_norm_odd = shapiro(p300_odd_amp[:min(len(p300_odd_amp), 5000)])
print('shapiro-wilk: p(std)=', round(p_norm_std, 4),
      'p(odd)=', round(p_norm_odd, 4))

alpha_norm = 0.05
if p_norm_std < alpha_norm or p_norm_odd < alpha_norm:
    print('normality violated -> mann-whitney U')
    stat, p_val = mannwhitneyu(p300_odd_amp, p300_std_amp, alternative='greater')
    test_name = 'Mann-Whitney U'
    df_val = None
    n1, n2 = len(p300_odd_amp), len(p300_std_amp)
    r_rb = 1 - (2 * stat) / (n1 * n2)
    d_val = r_rb
    d_label = 'r (rank-biserial)'
else:
    print('normality ok -> independent t-test')
    t_stat, p_val = ttest_ind(p300_odd_amp, p300_std_amp, alternative='greater')
    stat = t_stat
    df_val = len(p300_odd_amp) + len(p300_std_amp) - 2
    pooled_sd = np.sqrt(((len(p300_std_amp)-1)*p300_std_amp.std()**2 +
                         (len(p300_odd_amp)-1)*p300_odd_amp.std()**2) / df_val)
    d_val = (p300_odd_amp.mean() - p300_std_amp.mean()) / pooled_sd
    d_label = "Cohen's d"
    test_name = 'Independent-samples t-test'

print('test:', test_name)
print('statistic=', round(stat, 4))
if df_val:
    print('df=', df_val)
print('p=', round(p_val, 4))
print(d_label, '=', round(d_val, 4))

p300_results = {
    'M_std': p300_std_amp.mean(), 'SD_std': p300_std_amp.std(),
    'M_odd': p300_odd_amp.mean(), 'SD_odd': p300_odd_amp.std(),
    'stat': stat, 'df': df_val, 'p': p_val,
    'd': d_val, 'd_label': d_label, 'test': test_name,
    'N_std': len(p300_std_amp), 'N_odd': len(p300_odd_amp),
}


### APA write-up (from p300_results)

In [17]:
r = p300_results
df_str = '(' + str(int(r['df'])) + ')' if r['df'] else ''
if r['p'] < 0.001:
    p_str = '< .001'
else:
    p_str = '= ' + str(round(r['p'], 3)).replace('0.', '.')
d_str = str(round(abs(r['d']), 2))

print('P300 result (APA):')
print('M(odd)=', round(r['M_odd'], 2), 'SD=', round(r['SD_odd'], 2))
print('M(std)=', round(r['M_std'], 2), 'SD=', round(r['SD_std'], 2))
print('statistic' + df_str + '=', round(r['stat'], 2),
      'p', p_str, ',', r['d_label'], '=', d_str)


### P300 - what it means

larger P300 to oddball is the standard auditory oddball result, lines up with the context-updating account
of P300 generation (Donchin & Coles, 1988; Polich, 2007). The oddball is rare (~15%) so each one is
more "surprising" - P300 amplitude scales inversely with stimulus probability and directly with task
relevance (Polich, 2007). The centro-parietal max at Pz is consistent with hippocampal/parieto-frontal
generators tied to memory encoding and attention. The ~400 ms latency means this is higher-order
cognitive processing, not primary sensory.


## N200

RQ: does the oddball elicit a larger (more negative) N200 than the standard at Fz?

H0: no difference in N200 mean amplitude (180-280 ms, Fz).
H1: oddball N200 at Fz is more negative than standard (one-tailed).


### waveforms at Fz, Cz, FCz

In [18]:
n200_channels = [ch for ch in ['Fz', 'Cz', 'FCz'] if ch in evoked_std.ch_names]
print('plotting N200 at:', n200_channels)

fig, axes = plt.subplots(1, len(n200_channels), figsize=(6 * len(n200_channels), 5),
                          sharey=True)
if len(n200_channels) == 1:
    axes = [axes]

for ax, ch in zip(axes, n200_channels):
    idx_s = evoked_std.ch_names.index(ch)
    idx_o = evoked_odd.ch_names.index(ch)

    ax.plot(times_ep, evoked_std.data[idx_s] * 1e6,
            color=C_STD, lw=2, label='Standard')
    ax.plot(times_ep, evoked_odd.data[idx_o] * 1e6,
            color=C_ODD, lw=2, label='Oddball')

    ax.axvspan(180, 280, alpha=0.12, color='lightblue', label='N200 window')
    ax.axvline(0, color='k', lw=1, linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_xlabel('Time (ms)', fontsize=12)
    ax.set_ylabel('Amplitude (uV)', fontsize=12)
    ax.set_title('Channel ' + ch, fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(-200, 800)
    ax.invert_yaxis()  # negative up, ERP convention

fig.suptitle('N200 ERP Waveforms: Standard vs Oddball', fontsize=15, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '06_N200_erp_waveforms.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 06_N200_erp_waveforms.png')


### topomaps at peak latency

In [19]:
# most negative point in 180-280 ms at Fz
fz_idx = evoked_odd.ch_names.index('Fz')
t_mask_n200 = (evoked_odd.times >= 0.180) & (evoked_odd.times <= 0.280)
n200_data_fz = evoked_odd.data[fz_idx, t_mask_n200]
n200_peak_idx = np.argmin(n200_data_fz)
n200_peak_t = evoked_odd.times[t_mask_n200][n200_peak_idx]
print('N200 peak latency at Fz:', round(n200_peak_t * 1000, 1), 'ms')
print('N200 peak amplitude (oddball):',
      round(evoked_odd.data[fz_idx, t_mask_n200][n200_peak_idx]*1e6, 2), 'uV')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vlim_n2 = 4.0  # uV

for ax, evk, label in zip(axes, [evoked_std, evoked_odd], ['Standard', 'Oddball']):
    mne.viz.plot_topomap(
        evk.data[:, np.argmin(np.abs(evk.times - n200_peak_t))] * 1e6,
        evk.info, axes=ax, show=False, vlim=(-vlim_n2, vlim_n2),
        cmap='RdBu_r', contours=6
    )
    ax.set_title(label + '\n(t = ' + str(int(round(n200_peak_t*1000))) + ' ms)',
                 fontsize=12)

fig.suptitle('N200 Topography at Peak Latency', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '07_N200_topography.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 07_N200_topography.png')


### single-trial N200 amplitude + stats

In [20]:
fz_std_data = epochs_std.copy().pick('Fz').crop(0.180, 0.280).get_data()
fz_odd_data = epochs_odd.copy().pick('Fz').crop(0.180, 0.280).get_data()

n200_std_amp = fz_std_data[:, 0, :].mean(axis=1) * 1e6
n200_odd_amp = fz_odd_data[:, 0, :].mean(axis=1) * 1e6

print('standard: M=', round(n200_std_amp.mean(), 3),
      'SD=', round(n200_std_amp.std(), 3),
      'N=', len(n200_std_amp))
print('oddball:  M=', round(n200_odd_amp.mean(), 3),
      'SD=', round(n200_odd_amp.std(), 3),
      'N=', len(n200_odd_amp))

_, p_n_std = shapiro(n200_std_amp[:min(len(n200_std_amp), 5000)])
_, p_n_odd = shapiro(n200_odd_amp[:min(len(n200_odd_amp), 5000)])
print('shapiro-wilk: p(std)=', round(p_n_std, 4), 'p(odd)=', round(p_n_odd, 4))

if p_n_std < 0.05 or p_n_odd < 0.05:
    print('normality violated -> mann-whitney U (one-tailed, std > odd)')
    stat_n2, p_n2 = mannwhitneyu(n200_std_amp, n200_odd_amp, alternative='greater')
    test_n2 = 'Mann-Whitney U'
    df_n2 = None
    n1, n2 = len(n200_odd_amp), len(n200_std_amp)
    d_n2 = 1 - (2 * stat_n2) / (n1 * n2)
    d_label_n2 = 'r (rank-biserial)'
else:
    print('normality ok -> independent t-test')
    t_stat_n2, p_n2 = ttest_ind(n200_std_amp, n200_odd_amp, alternative='greater')
    stat_n2 = t_stat_n2
    df_n2 = len(n200_std_amp) + len(n200_odd_amp) - 2
    pooled = np.sqrt(((len(n200_std_amp)-1)*n200_std_amp.std()**2 +
                       (len(n200_odd_amp)-1)*n200_odd_amp.std()**2) / df_n2)
    d_n2 = (n200_std_amp.mean() - n200_odd_amp.mean()) / pooled
    d_label_n2 = "Cohen's d"
    test_n2 = 'Independent-samples t-test'

print('test:', test_n2)
print('statistic=', round(stat_n2, 4))
if df_n2:
    print('df=', df_n2)
print('p=', round(p_n2, 4))
print(d_label_n2, '=', round(d_n2, 4))

n200_results = {
    'M_std': n200_std_amp.mean(), 'SD_std': n200_std_amp.std(),
    'M_odd': n200_odd_amp.mean(), 'SD_odd': n200_odd_amp.std(),
    'stat': stat_n2, 'df': df_n2, 'p': p_n2,
    'd': d_n2, 'd_label': d_label_n2, 'test': test_n2,
}


In [21]:
r2 = n200_results
df2_str = '(' + str(int(r2['df'])) + ')' if r2['df'] else ''
if r2['p'] < 0.001:
    p2_str = '< .001'
else:
    p2_str = '= ' + str(round(r2['p'], 3)).replace('0.', '.')
d2_str = str(round(abs(r2['d']), 2))

print('N200 result (APA):')
print('M(odd)=', round(r2['M_odd'], 2), 'SD=', round(r2['SD_odd'], 2))
print('M(std)=', round(r2['M_std'], 2), 'SD=', round(r2['SD_std'], 2))
print('statistic' + df2_str + '=', round(r2['stat'], 2),
      'p', p2_str, ',', r2['d_label'], '=', d2_str)


### N200 - what it means

frontocentral N200 reflects pre-attentive deviance detection (Naatanen et al., 2007).
two things going on here:
1. it's basically the MMN - automatic auditory change detection that runs without attention.
2. since this task needs a button press to oddballs, there's also an N2b component (~200-300 ms)
   that's more about attentional orienting, followed by P3a/P3b downstream.

so N200 + P300 enhancement together = the full hierarchy from automatic change detection (N200/MMN)
to controlled updating + working memory revision (P300).


## MMN difference wave

oddball minus standard. cancels the shared auditory response and leaves the change-detection signature.
typically peaks 100-250 ms over frontocentral, inverts at the mastoids - generators in auditory cortex.


In [22]:
diff_evoked = mne.combine_evoked([evoked_odd, evoked_std], weights=[1, -1])
diff_evoked.comment = 'Oddball - Standard (difference wave)'

fz_idx_d = diff_evoked.ch_names.index('Fz')
fz_data_d = diff_evoked.data[fz_idx_d] * 1e6

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(times_ep, fz_data_d, color=C_DIFF, lw=2.5, label='Oddball - Standard')
ax.axvspan(150, 250, alpha=0.12, color='mediumpurple', label='MMN window (150-250 ms)')
ax.axvline(0, color='k', lw=1.2, linestyle='--', alpha=0.8)
ax.axhline(0, color='k', lw=0.5, alpha=0.5)
ax.fill_between(times_ep, fz_data_d, 0,
                where=(fz_data_d < 0) & (times_ep >= 150) & (times_ep <= 250),
                color=C_DIFF, alpha=0.25, label='Negative MMN region')
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Amplitude (uV)', fontsize=12)
ax.set_title('MMN Difference Wave at Fz (Oddball - Standard)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-200, 800)
ax.invert_yaxis()

# annotate the peak
t_mmn = (times_ep >= 150) & (times_ep <= 250)
mmn_peak_amp = fz_data_d[t_mmn].min()
mmn_peak_lat = times_ep[t_mmn][np.argmin(fz_data_d[t_mmn])]
ax.annotate('MMN peak\n' + str(int(round(mmn_peak_lat))) + ' ms, ' +
            str(round(mmn_peak_amp, 2)) + ' uV',
            xy=(mmn_peak_lat, mmn_peak_amp),
            xytext=(mmn_peak_lat + 50, mmn_peak_amp - 0.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '08_MMN_difference_wave.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 08_MMN_difference_wave.png')
print('MMN peak:', round(mmn_peak_lat, 1), 'ms,', round(mmn_peak_amp, 2), 'uV at Fz')


### MMN interpretation

frontocentral negative deflection that pops up automatically - even during sleep or distraction -
when an auditory deviant breaks a regular sequence (Naatanen et al., 2007). generators are
primary/secondary auditory cortex (slightly right-dominant) and SMA.

theoretically: auditory cortex maintains a predictive model of recent stimulus history. when the
oddball violates the prediction, the prediction error shows up as the MMN. lines up well with
predictive coding accounts of perception.


## extra credit: reference comparison

trying four references and comparing the P300 topography at peak latency:
1. average (mean of all electrodes - good when coverage is uniform)
2. Cz (vertex)
3. linked mastoids - using TP7 + TP8 as the closest available channels to M1/M2 in this biosemi cap
4. Fpz (nasion proxy)


In [23]:
mastoid_candidates = ['TP9', 'TP10', 'M1', 'M2', 'TP7', 'TP8', 'P9', 'P10', 'T7', 'T8']
avail_ch = raw_before_reref.copy().pick('eeg').ch_names
mastoid_ch = [c for c in mastoid_candidates if c in avail_ch][:2]
print('mastoid proxy channels:', mastoid_ch)

def make_epochs_with_ref(raw_in, ref_channels, scheme_name):
    r = raw_in.copy().pick('eeg')
    r.set_eeg_reference(ref_channels, projection=False, verbose=False)
    r.filter(l_freq=0.1, h_freq=30.0, picks='eeg', verbose=False)
    ep = mne.Epochs(r, events_array, event_id={'oddball': 2},
                    tmin=-0.2, tmax=0.8, baseline=(-0.2, 0.0),
                    reject=dict(eeg=100e-6), preload=True, verbose=False)
    ep.drop_bad(verbose=False)
    evk = ep.average()
    print(' ', scheme_name, ':', len(ep), 'epochs')
    return evk

print('building evokeds for each reference scheme...')
evk_avg = make_epochs_with_ref(raw_before_reref, 'average', 'Average reference')
evk_cz = make_epochs_with_ref(raw_before_reref, ['Cz'], 'Cz reference')
evk_mast = make_epochs_with_ref(raw_before_reref, mastoid_ch,
                                  'Linked mastoids (' + '+'.join(mastoid_ch) + ')')
evk_fpz = make_epochs_with_ref(raw_before_reref, ['Fpz'], 'Fpz reference')

ref_evokeds = [
    (evk_avg, 'Average reference'),
    (evk_cz, 'Cz reference'),
    (evk_mast, 'Linked mastoids\n(' + '+'.join(mastoid_ch) + ')'),
    (evk_fpz, 'Fpz reference'),
]
print('done')


In [24]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('P300 Topography at ' + str(int(round(p300_peak_t*1000))) +
             ' ms - Reference Comparison',
             fontsize=14, fontweight='bold')

vlim_ref = 6.0  # uV common color scale

for ax, (evk, label) in zip(axes, ref_evokeds):
    t_idx = np.argmin(np.abs(evk.times - p300_peak_t))
    topo_data = evk.data[:, t_idx] * 1e6
    im, _ = mne.viz.plot_topomap(
        topo_data, evk.info, axes=ax, show=False,
        vlim=(-vlim_ref, vlim_ref), cmap='RdBu_r', contours=6
    )
    ax.set_title(label, fontsize=11)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='RdBu_r',
                            norm=plt.Normalize(-vlim_ref, vlim_ref))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label='uV')

plt.subplots_adjust(right=0.90, wspace=0.05)
fig.savefig(os.path.join(FIG_DIR, '09_reference_comparison_topos.png'),
            dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved 09_reference_comparison_topos.png')


### why reference matters

EEG always measures a difference: V_i = phi_i - phi_r. changing the reference is just subtracting a
constant offset from all channels - so no reference is "neutral", every choice biases activity
toward zero at the reference site.

what i see in the four maps:
- average reference: balanced centro-parietal positivity, P300 cleanly isolated. best overall.
- Cz reference: P300 looks suppressed at Pz because Cz itself is active during the response,
  so subtracting it pulls down neighboring channels.
- linked mastoids: parietal positivity looks stronger, good SNR for auditory ERPs, but bilateral
  temporal bias.
- Fpz reference: severe frontal distortion, P300 spreads anteriorly.

for this 64-channel biosemi cap with uniform-ish coverage the average reference is probably the
best default (Dien, 1998; Yao, 2001). linked mastoids would be a fine alternative if i specifically
cared about occipito-temporal sensors.


## references

Dien, J. (1998). Issues in the application of the average reference: Review, critiques, and
recommendations. *Behavior Research Methods, Instruments, & Computers, 30*(1), 34-43.

Donchin, E., & Coles, M. G. H. (1988). Is the P300 component a manifestation of context updating?
*Behavioral and Brain Sciences, 11*(3), 357-374.

Luck, S. J. (2014). *An Introduction to the Event-Related Potential Technique* (2nd ed.). MIT Press.

Naatanen, R., Paavilainen, P., Rinne, T., & Alho, K. (2007). The mismatch negativity (MMN) in
basic research of central auditory processing. *Clinical Neurophysiology, 118*(12), 2544-2590.

Polich, J. (2007). Updating P300: An integrative theory of P3a and P3b. *Clinical Neurophysiology,
118*(10), 2128-2148.

Yao, D. (2001). A method to standardize a reference of scalp EEG recording to a point at infinity.
*Physiological Measurement, 22*(4), 693-711.
